In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [20]:
df = pd.read_csv("../data/raw/House_Prices.csv")
df = df.iloc[:1460].copy()

In [ ]:
# remove outliers that has higher GrLivArea with less saleprice
df = df.drop(index=[1298, 523])

df = df.drop(columns="Id")
df["MSSubClass"] = df["MSSubClass"].astype("str")

# fill missing values in categorical columns with none
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageQual", "GarageCond", "GarageFinish", "GarageType",
    "BsmtCond", "BsmtQual", "BsmtFinType1"
]
df[none_cols] = df[none_cols].fillna("None")

# fill missing values in num columns with 0
zero_cols = [
    "BsmtFullBath", "BsmtHalfBath", "BsmtFinSF1",
    "GarageArea", "GarageCars"
]
df[zero_cols] = df[zero_cols].fillna(0)

# if the garage built year greater than the sold year mark the values as missing
df.loc[df["GarageYrBlt"] > df["YrSold"], "GarageYrBlt"] = np.nan

In [22]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

In [23]:
X.shape, y.shape

((1460, 79), (1460,))

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

In [25]:
X_train.shape, X_test.shape

((1168, 79), (292, 79))

In [26]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

In [27]:
print("Numerical:", len(numeric_cols))
print("Categorical:", len(categorical_cols))

Numerical: 35
Categorical: 44


In [28]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [29]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [30]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [31]:
X_train_processed = preprocessor.fit_transform(X_train)

In [32]:
X_test_processed = preprocessor.transform(X_test)

In [33]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(1168, 299)
(292, 299)


In [34]:
print("Train missing values:", X_train.isna().sum().sum())
print("Test missing values:", X_test.isna().sum().sum())

Train missing values: 6227
Test missing values: 1602


In [35]:
type(X_train_processed)

scipy.sparse._csr.csr_matrix

In [36]:
import numpy as np
print("Train missing values:", np.isnan(X_train_processed.data).sum())
print("Test missing values:", np.isnan(X_test_processed.data).sum())

Train missing values: 0
Test missing values: 0
